# LA Studio Qwen3 Forced Alignment GPU Worker

Choose **Runtime > Change runtime type > GPU**, then **Run all**. This notebook starts a temporary direct Colab worker for Qwen3 ForcedAligner. It accepts only the audio file and transcript sent by LA Studio and does not read, store, call, or proxy an API Gateway key. The final cell prints the HTTPS URL and temporary bearer token for the **Direct Colab Alignment** settings panel.

In [ ]:
import subprocess, sys

def run(*args):
    print('+', ' '.join(args))
    subprocess.run(args, check=True)

run('nvidia-smi')
run('ffprobe', '-version')
run(sys.executable, '-m', 'pip', 'install', '--quiet', 'qwen-asr', 'fastapi>=0.115.0', 'uvicorn[standard]>=0.34.0', 'python-multipart>=0.0.20')


In [ ]:
from pathlib import Path

WORKER = Path('/content/la_studio_alignment_worker.py')
WORKER.write_text(r'''
import os
import re
import subprocess
import tempfile
import threading
from pathlib import Path

import torch
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from qwen_asr import Qwen3ForcedAligner

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Select a GPU runtime before starting this worker.')

TOKEN = os.environ['LA_STUDIO_COLAB_TOKEN']
MODEL_ID = 'Qwen/Qwen3-ForcedAligner-0.6B'
SUPPORTED_MODELS = {'qwen3-forced-aligner-0.6b', 'qwen/qwen3-forcedaligner-0.6b'}
LANGUAGES = {'zh': 'Chinese', 'en': 'English', 'yue': 'Cantonese', 'fr': 'French', 'de': 'German', 'it': 'Italian', 'ja': 'Japanese', 'ko': 'Korean', 'pt': 'Portuguese', 'ru': 'Russian', 'es': 'Spanish'}
MODEL_LOCK = threading.Lock()
MAX_UPLOAD_BYTES = 512 * 1024 * 1024
MAX_AUDIO_SECONDS = 300
ALLOWED_CONTENT_TYPES = {'audio/wav', 'audio/x-wav', 'audio/mpeg', 'audio/mp4', 'audio/webm', 'audio/ogg', 'audio/flac'}
REQUEST_SLOTS = threading.BoundedSemaphore(1)

def require_token(authorization: str | None) -> None:
    if authorization != 'Bearer ' + TOKEN:
        raise HTTPException(status_code=401, detail='invalid worker token')

def media_duration_seconds(path: str) -> float:
    probe = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'default=nokey=1:noprint_wrappers=1', path], text=True, capture_output=True)
    try:
        duration = float(probe.stdout.strip())
    except ValueError:
        duration = 0.0
    if probe.returncode != 0 or duration <= 0.0:
        raise HTTPException(status_code=415, detail='audio is unsupported or could not be decoded')
    return duration

def attr(item, *names, default=None):
    for name in names:
        value = getattr(item, name, None)
        if value is not None:
            return value
        if isinstance(item, dict) and item.get(name) is not None:
            return item[name]
    return default

def flatten(items):
    for item in items if isinstance(items, (list, tuple)) else [items]:
        nested = attr(item, 'items', 'segments', 'words')
        text = attr(item, 'text', 'token', 'word', default='')
        start = attr(item, 'start_time', 'start', 'begin')
        end = attr(item, 'end_time', 'end', 'finish')
        if nested is not None and (start is None or end is None):
            yield from flatten(nested)
        elif str(text).strip() and start is not None and end is not None:
            yield {'text': str(text).strip(), 'start': float(start), 'end': float(end),
                   'score': float(attr(item, 'score', 'confidence', default=0.0) or 0.0),
                   'kind': 'token'}

def best_effort_unaligned(transcript: str, segments: list[dict], language: str) -> list[str]:
    # Word comparison is meaningful for whitespace-delimited languages only.
    if language in {'zh', 'yue', 'ja', 'ko'}:
        return []
    expected = re.findall(r"[^\W_]+(?:['-][^\W_]+)?", transcript.lower(), flags=re.UNICODE)
    aligned = [str(item['text']).lower() for item in segments]
    missing, cursor = [], 0
    for token in expected:
        while cursor < len(aligned) and aligned[cursor] != token:
            cursor += 1
        if cursor == len(aligned):
            missing.append(token)
        else:
            cursor += 1
    return missing

model = Qwen3ForcedAligner.from_pretrained(MODEL_ID, dtype=torch.float16, device_map='cuda:0')
app = FastAPI(title='LA Studio Colab Alignment Worker', docs_url=None, redoc_url=None, openapi_url=None)

@app.get('/health')
@app.get('/v1/health')
def health(authorization: str | None = Header(default=None)):
    require_token(authorization)
    return {'status': 'ready', 'ready': True, 'device': 'cuda', 'gpu': torch.cuda.get_device_name(0), 'api_version': '1.0'}

@app.get('/v1/capabilities')
def capabilities(authorization: str | None = Header(default=None)):
    require_token(authorization)
    return {'contract_version': 1, 'capabilities': [{'id': 'forced-alignment', 'models': [{'id': 'qwen3-forced-aligner-0.6b', 'upstream_model': MODEL_ID, 'languages': sorted(LANGUAGES), 'max_audio_seconds': 300, 'device': 'cuda'}]}]}

@app.post('/v1/audio/alignments')
async def align(audio: UploadFile = File(...), transcript: str = Form(...), language: str = Form('en'), model: str = Form(...), authorization: str | None = Header(default=None)):
    require_token(authorization)
    if model.strip().lower() not in SUPPORTED_MODELS:
        raise HTTPException(status_code=422, detail='this worker supports qwen3-forced-aligner-0.6b only')
    language_code = language.strip().lower()
    language_name = LANGUAGES.get(language_code)
    if not language_name:
        raise HTTPException(status_code=422, detail='unsupported Qwen3 ForcedAligner language')
    text = transcript.strip()
    if not text:
        raise HTTPException(status_code=422, detail='transcript is required')
    if audio.content_type not in ALLOWED_CONTENT_TYPES:
        raise HTTPException(status_code=415, detail='unsupported audio MIME type')
    suffix = Path(audio.filename or 'audio.wav').suffix.lower() or '.wav'
    if suffix not in {'.wav', '.mp3', '.m4a', '.mp4', '.webm', '.ogg', '.flac'}:
        raise HTTPException(status_code=415, detail='unsupported audio filename extension')
    if not REQUEST_SLOTS.acquire(blocking=False):
        raise HTTPException(status_code=429, detail='the Colab alignment worker is busy; retry shortly')
    source_path = None
    try:
        descriptor, source_path = tempfile.mkstemp(suffix=suffix)
        with os.fdopen(descriptor, 'wb') as source:
            while chunk := await audio.read(1024 * 1024):
                source.write(chunk)
                if source.tell() > MAX_UPLOAD_BYTES:
                    raise HTTPException(status_code=413, detail='audio exceeds 512 MB upload limit')
        duration_seconds = media_duration_seconds(source_path)
        if duration_seconds > MAX_AUDIO_SECONDS:
            raise HTTPException(status_code=413, detail='audio exceeds the five minute duration limit')
        with MODEL_LOCK, torch.inference_mode():
            raw_results = model.align(audio=source_path, text=text, language=language_name)
        segments = list(flatten(raw_results))
        if not segments:
            raise HTTPException(status_code=422, detail='the aligner returned no timestamped tokens')
        segments.sort(key=lambda item: (item['start'], item['end']))
        previous_end = 0.0
        for item in segments:
            if item['start'] < 0 or item['end'] < item['start'] or item['start'] + 0.002 < previous_end:
                raise HTTPException(status_code=502, detail='aligner returned non-monotonic timestamps')
            item['score'] = max(0.0, min(1.0, item['score']))
            previous_end = item['end']
        return {'duration': duration_seconds, 'segments': segments, 'unaligned_tokens': best_effort_unaligned(text, segments, language_code)}
    finally:
        try:
            if source_path:
                os.unlink(source_path)
        except FileNotFoundError:
            pass
        await audio.close()
        REQUEST_SLOTS.release()
''')
print('Worker source written:', WORKER)


In [ ]:
import os, re, secrets, subprocess, time, urllib.request

TOKEN = secrets.token_urlsafe(32)
env = os.environ.copy()
env['LA_STUDIO_COLAB_TOKEN'] = TOKEN
worker = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'la_studio_alignment_worker:app', '--host', '127.0.0.1', '--port', '3923'], cwd='/content', env=env)
for _ in range(45):
    try:
        request = urllib.request.Request('http://127.0.0.1:3923/health', headers={'Authorization': 'Bearer ' + TOKEN})
        with urllib.request.urlopen(request, timeout=3) as response:
            if response.status == 200:
                break
    except Exception:
        time.sleep(2)
else:
    worker.terminate()
    raise RuntimeError('LA Studio alignment worker did not become ready')
subprocess.run(['bash', '-lc', 'wget -q -O /content/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i /content/cloudflared.deb'], check=True)
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:3923', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for _ in range(90):
    line = tunnel.stdout.readline()
    print(line, end='')
    match = re.search(r'https://[^\s]+trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
if not public_url:
    worker.terminate(); tunnel.terminate()
    raise RuntimeError('Cloudflare tunnel URL was not found')
print('\nLA_STUDIO_COLAB_ALIGNMENT_URL=' + public_url)
print('LA_STUDIO_COLAB_ALIGNMENT_TOKEN=' + TOKEN)
print('MODEL=qwen3-forced-aligner-0.6b')


In [ ]:
# ==============================================================================
# 📥 LƯU FILE TRỰC TIẾP VÀO THƯ MỤC DỰ ÁN TRÊN MÁY TÍNH (FILE SYSTEM ACCESS API)
# ==============================================================================
import base64
import glob
import json
import os
from IPython.display import HTML, display

# Thu thập tất cả các file kết quả vừa tạo
result_files = {}
for pattern in ['/content/*.wav', '/content/*.srt', '/content/*.json', '/content/*/*/*.wav', '/content/*/*/*.srt']:
    for f in glob.glob(pattern):
        name = os.path.basename(f)
        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:
            with open(f, 'rb') as fp:
                result_files[name] = base64.b64encode(fp.read()).decode('utf-8')

if not result_files:
    print("⚠️ Chưa có file kết quả mới để lưu.")
else:
    print(f"✅ Đã tìm thấy {len(result_files)} file kết quả: {', '.join(result_files.keys())}")
    print("👉 Bấm nút bên dưới và chọn thư mục 'LA-Studio/out/colab-live' để lưu thẳng vào máy:")
    
    files_json = json.dumps(result_files)
    html_code = f"""
    <button id="saveBtn" style="background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;">
        📁 Chọn Thư Mục & Lưu File Trực Tiếp Vào Máy
    </button>
    <div id="statusLog" style="margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;"></div>
    <script>
    document.getElementById('saveBtn').onclick = async () => {{
        const log = document.getElementById('statusLog');
        try {{
            if (!window.showDirectoryPicker) {{
                log.innerText = 'Trình duyệt không hỗ trợ File System Access API. Đang dùng tải thông thường...';
                return;
            }}
            log.innerText = 'Đang mở hộp thoại chọn thư mục...';
            const dirHandle = await window.showDirectoryPicker();
            const files = {files_json};
            for (const [name, b64] of Object.entries(files)) {{
                log.innerText = 'Đang ghi file: ' + name + '...';
                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});
                const writable = await fileHandle.createWritable();
                const byteCharacters = atob(b64);
                const byteNumbers = new Array(byteCharacters.length);
                for (let i = 0; i < byteCharacters.length; i++) {{
                    byteNumbers[i] = byteCharacters.charCodeAt(i);
                }}
                const byteArray = new Uint8Array(byteNumbers);
                await writable.write(byteArray);
                await writable.close();
            }}
            log.innerText = '🎉 Đã lưu thành công toàn bộ file vào thư mục bạn chọn!';
        }} catch (err) {{
            if (err.name !== 'AbortError') {{
                log.innerText = 'Lỗi: ' + err.message;
            }} else {{
                log.innerText = 'Đã hủy chọn thư mục.';
            }}
        }}
    }};
    </script>
    """
    display(HTML(html_code))
